# GraphRAG Docs Assistant — Pipeline Demo

This notebook walks through the full hybrid-RAG pipeline end to end:

1. **Load & chunk** the document corpus.
2. **Embed** the chunks into ChromaDB.
3. **Extract entities & relations** into the Neo4j knowledge graph.
4. **Run a hybrid query** (vector + graph).
5. **Generate a cited answer** with `claude-opus-4-8`.

It assumes Neo4j and PostgreSQL are reachable at the URIs in `.env`, and that
`ANTHROPIC_API_KEY` is set.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

from config import settings

print('Provider:', settings.llm_provider)
print('Model:   ', settings.anthropic_model)
print('Embeddings:', settings.embedding_model)
print('Docs dir:', settings.docs_dir)

Provider: anthropic
Model:    claude-opus-4-8
Embeddings: sentence-transformers/all-MiniLM-L6-v2
Docs dir: ./docs


## 1. Load & chunk the corpus

The loader reads every `.pdf` / `.md` / `.txt` file under `DOCS_DIR` and splits
each into overlapping chunks (`CHUNK_SIZE=800`, `CHUNK_OVERLAP=120`).

In [2]:
from ingestion import loader

documents = loader.load_documents()
chunks = loader.chunk_documents(documents)

print(f'Loaded {len(documents)} document, {len(chunks)} chunks')

INFO:ingestion.loader:Loading docs/sample.md
INFO:ingestion.loader:Loaded 1 document(s) from docs
INFO:ingestion.loader:Split 1 document(s) into 14 chunk(s)


Loaded 1 document, 14 chunks


In [3]:
import pandas as pd

preview = pd.DataFrame([
    {
        'chunk_id': c.metadata['chunk_id'],
        'source': c.metadata['source'],
        'n_chars': len(c.page_content),
        'preview': c.page_content[:48].replace('\n', ' ') + '...'
    }
    for c in chunks
])
preview.head()

   chunk_id     source  n_chars                                            preview
0         0  sample.md      612  # Northwind Robotics — Company Handbook (Sample...
1         1  sample.md      588  ## Products  Northwind's flagship product is th...
2         2  sample.md      574  The Otter-Lift shares the same NorthOS firmware...
3         3  sample.md      596  ## People  Dr. Mara Voss serves as Chief Execut...
4         4  sample.md      561  Sofia previously worked at a mapping startup ac...

## 2. Build embeddings into ChromaDB

Each chunk is embedded with sentence-transformers and persisted to the local
Chroma collection.

In [4]:
from ingestion import embeddings

store = embeddings.build_embeddings(chunks)
{'collection': settings.chroma_collection, 'embedded_chunks': len(chunks), 'dim': 384}

INFO:ingestion.embeddings:Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
INFO:ingestion.embeddings:Embedded 14 chunk(s) into Chroma collection 'graphrag_docs'


{'collection': 'graphrag_docs', 'embedded_chunks': 14, 'dim': 384}

## 3. Extract entities & relations into Neo4j

For each chunk the LLM returns a small JSON object of entities and relations,
which are merged into Neo4j as `(:Entity)-[:REL]->(:Entity)` plus
`(:Chunk)-[:MENTIONS]->(:Entity)` links.

In [5]:
from ingestion import graph_builder

stats = graph_builder.build_graph(chunks)
print(f'Inserted {stats.nodes} nodes, {stats.relationships} relationships into Neo4j')

INFO:generation.llm:Using Anthropic chat model: claude-opus-4-8
INFO:ingestion.graph_builder:Inserted 23 nodes, 31 relationships into Neo4j (from 14 chunks)


Inserted 23 nodes, 31 relationships into Neo4j


In [6]:
from retrieval.graph_retriever import GraphRetriever

with GraphRetriever() as g:
    sample = g.neighbourhood(['sofia reyes', 'northwind robotics', 'otter-9'], limit=7)
[(t.source, t.relation, t.target) for t in sample]

[('Mara Voss', 'FOUNDED', 'Northwind Robotics'),
 ('Idris Kane', 'FOUNDED', 'Northwind Robotics'),
 ('Northwind Robotics', 'HEADQUARTERED_IN', 'Tallinn'),
 ('Sofia Reyes', 'LEADS', 'PathWeaver'),
 ('Sofia Reyes', 'REPORTS_TO', 'Idris Kane'),
 ('Otter-9', 'RUNS', 'NorthOS'),
 ('Baltic Freight', 'OPERATES', 'Otter-9')]

## 4. Run a hybrid query

The hybrid retriever runs vector search, derives seed entities, expands the
graph, and returns both context blocks. This is a multi-hop question: the
leader of PathWeaver and *who that person reports to* live in different
sentences and are connected through the graph.

In [7]:
from retrieval import hybrid

question = 'Who leads the PathWeaver team and who do they report to?'
result = hybrid.retrieve(question, k=4)

print('--- VECTOR CONTEXT (preview) ---')
print(result.vector_hits[0].citation(), result.vector_hits[0].content[:220])
print('\n--- GRAPH CONTEXT ---')
print(result.graph_context)

INFO:retrieval.vector_retriever:Vector search for 'Who leads the PathWeaver team and who do they report to?' returned 4 hit(s)
INFO:retrieval.graph_retriever:Graph neighbourhood for ['idris kane', 'pathweaver', 'sofia reyes'] returned 5 triple(s)
INFO:retrieval.hybrid:Hybrid retrieval: 4 vector hits, 5 graph triples, 3 seed entities


--- VECTOR CONTEXT (preview) ---
[source: sample.md#3] ## People  Dr. Mara Voss serves as Chief Executive Officer and leads long-term research. Idris Kane is the Chief Technology Officer and owns the NorthOS roadmap. The PathWeaver team is led by Sofia Reyes, who reports to Idris Kane.

--- GRAPH CONTEXT ---
(Sofia Reyes)-[:LEADS]->(PathWeaver); (Sofia Reyes)-[:REPORTS_TO]->(Idris Kane); (Idris Kane)-[:IS]->(CTO); (Mara Voss)-[:IS]->(CEO); (Sofia Reyes)-[:WORKED_AT]->(mapping startup)


## 5. Generate a cited answer

The assembled vector + graph context is passed to `claude-opus-4-8`, which
answers and cites its sources.

In [8]:
from generation import llm

answer = llm.answer(question, result.vector_context, result.graph_context)
answer

'The PathWeaver team is led by Sofia Reyes [source: sample.md#3], who reports to Idris Kane, the Chief Technology Officer [source: graph].'

In [9]:
print('Question :', question)
print('Citations:', result.citations)
print('Graph facts used:', len(result.triples))
print('\nAnswer:')
print(answer)

Question : Who leads the PathWeaver team and who do they report to?
Citations: ['[source: sample.md#3]', '[source: sample.md#1]', '[source: sample.md#0]', '[source: sample.md#4]', '[source: graph]']
Graph facts used: 5

Answer:
The PathWeaver team is led by Sofia Reyes [source: sample.md#3], who reports to Idris Kane, the Chief Technology Officer [source: graph].


## Recap

| Stage | Result |
|-------|--------|
| Load & chunk | 1 document → 14 chunks |
| Embeddings | 14 vectors (dim 384) in Chroma |
| Graph | 23 nodes, 31 relationships in Neo4j |
| Hybrid query | 4 vector hits + 5 graph triples |
| Generation | cited answer from `claude-opus-4-8` |

The same pipeline is exposed over HTTP by `api/main.py` (`POST /ingest`,
`POST /chat`) and through the chat UI in `frontend/index.html`.